In [ ]:
import sys
sys.path.append('../../../')
from pathlib import Path
import h5py
import numpy as np
import jax.numpy as jnp
import sys
import os
import torch

from lucid.geometry import generate_detector
from lucid.production.data_prod_utils import read_multi_event_file, print_event_info, get_particle_name, get_track_hits

figures_dir = Path('figures')
figures_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def visualize_detector_geometry(detector_name, colorscale='plasma', surface_color='black'):
    """
    Visualize detector geometry with all sensors having equal charge (same color)
    """
    json_filename = f'../../../config/{detector_name}_geom_config.json'
    detector = generate_detector(json_filename)
    
    # Create equal charges for all sensors (all = 1)
    n_sensors = len(detector.all_points)
    indices = jnp.arange(n_sensors)
    charges = jnp.ones(n_sensors)  # All charges equal to 1
    times = jnp.zeros(n_sensors)   # Times don't matter for geometry visualization
    
    print(f"Detector: {detector_name}")
    print(f"Number of sensors: {n_sensors}")
    
    # Create visualization
    figname = f'figures/{detector_name}_geometry.pdf'
    detector.visualize_event_data_plotly_discs(
        indices, charges, times, 
        show_all_sensors=True, 
        log_scale=False,  # No need for log scale with equal charges
        show_colorbar=False, 
        dark_theme=False, 
        plot_time=False, 
        colorscale=colorscale, 
        surface_color=surface_color, 
        figname=figname
    )
    
    return detector

In [ ]:
detector_name = 'SK'
try:
    print(f"\nVisualizing {detector_name}...")
    visualize_detector_geometry(detector_name, colorscale='plasma', surface_color='black')
    print(f"Saved: figures/{detector_name}_geometry.pdf")
except Exception as e:
    print(f"Error visualizing {detector_name}: {e}")

In [ ]:
def visualize_3D_data_event_for_detector(indices, charges, times, colorscale='viridis', surface_color='gray', log_scale=False, plot_time=False, figname=None):
    """Visualize data event using disc visualization"""

    json_filename = f'../../../config/SK_geom_config.json'
    detector = generate_detector(json_filename)
    
    detector.visualize_event_data_plotly_discs(
        indices, 
        charges, 
        times, 
        show_all_sensors=False, 
        log_scale=log_scale, 
        show_colorbar=False, 
        dark_theme=False, 
        plot_time=plot_time, 
        colorscale=colorscale,
        inactive_color='black',
        surface_color=surface_color, 
        figname=figname,
        title=None
    )

In [ ]:
filename = '/sdf/data/neutrino/cjesus/photonsim_output/water/uniform_energy/multiparticle/config_000004/events_job_000001.h5'
all_events = read_multi_event_file(filename, verbose=False)

In [ ]:
# Read a single event with detailed matrix display
event = read_multi_event_file(filename, event_index=0, verbose=True, show_matrices=True)

# You can also control how many PMTs to show
print("\n" + "="*80)
print("Custom matrix display with 20 PMTs:")
print("="*80)
print_event_info(event, title=None, show_matrices=True, n_pmts_to_show=20)

In [ ]:
event_idx = 0
indices, Q, T = get_track_hits(all_events[event_idx], 1)
visualize_3D_data_event_for_detector(indices, Q, T, colorscale='viridis', surface_color='lightgray', log_scale=True, plot_time=True)

In [ ]:
import matplotlib.cm as cm

import numpy as np
import matplotlib.pyplot as plt


def build_flat_colorscale(norm_overlap, norm_values, base_colors):
    """
    Builds a 'flat' colorscale with no interpolation between colors.
    Mesh3d requires repeated stops to prevent blending.
    """

    colorscale = []

    # Overlap (black)
    colorscale.append([norm_overlap, 'black'])
    colorscale.append([norm_values[0] - 1e-6, 'black'])

    # Particles
    for nv, col in zip(norm_values, base_colors):
        colorscale.append([nv - 1e-6, col])
        colorscale.append([nv, col])

    return colorscale



def prepare_event_visualization(event):
    """
    For an event, compute:
    - hits per track (PMTs where Q > 0)
    - overlap PMTs (hit by 2 or more tracks)
    - unique integers representing a color for each track
    - a dedicated overlap color (e.g., black)

    Returns
    -------
    particle_hits : list of arrays
        Each entry is the list of nonzero PMT indices for a track.
    Q_list : list of arrays
        Filtered Q values
    T_list : list of arrays
        Filtered T values
    overlap_indices : ndarray
        Sorted array of PMT indices hit by ≥ 2 tracks
    """
    n_tracks = event["Q"].shape[0]

    particle_hits = []
    Q_list = []
    T_list = []

    # Collect nonzero indices for each track
    all_indices = []

    for track_idx in range(n_tracks):
        idx, Q_f, T_f = get_track_hits(event, track_idx)
        particle_hits.append(idx)
        Q_list.append(Q_f)
        T_list.append(T_f)
        all_indices.extend(idx.tolist())

    # Find overlaps (PMTs hit by 2+ tracks)
    unique, counts = np.unique(all_indices, return_counts=True)
    overlap_indices = unique[counts > 1]

    print(len(particle_hits[0]), len(particle_hits[1]), np.shape(overlap_indices))
    return particle_hits, Q_list, T_list, overlap_indices


import numpy as np
import matplotlib

def visualize_event_all_particles(event):
    particle_hits, Q_list, T_list, overlap_indices = prepare_event_visualization(event)

    n_tracks = len(particle_hits)

    # Choose fixed colors (extend if needed)
    base_colors = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'magenta']
    while len(base_colors) < n_tracks:
        base_colors += base_colors

    # -----------------------------------------
    # BUILD NORMALIZED VALUES (0..1)
    # -----------------------------------------

    norm_overlap = 0.0
    norm_values = np.linspace(0.2, 1.0, n_tracks)
    
    colorscale = build_flat_colorscale(norm_overlap, norm_values, base_colors[:n_tracks])


    # Build colorscale for Plotly
    colorscale = [[norm_overlap, 'black']]
    for nv, col in zip(norm_values, base_colors[:n_tracks]):
        colorscale.append([float(nv), col])

    # -----------------------------------------
    # COLLECT ALL PMTs FOR SINGLE PLOT
    # -----------------------------------------

    all_indices = []
    all_norm_intensity = []
    all_times = []

    for pid, (idx, Q, T) in enumerate(zip(particle_hits, Q_list, T_list)):

        # Overlap mask
        is_overlap = np.isin(idx, overlap_indices)

        # Unique hits
        for ind, tval in zip(idx[~is_overlap], T[~is_overlap]):
            all_indices.append(ind)
            all_norm_intensity.append(norm_values[pid])   # normalized value → particle color
            all_times.append(tval)

        # Overlap hits
        for ind, tval in zip(idx[is_overlap], T[is_overlap]):
            all_indices.append(ind)
            all_norm_intensity.append(norm_overlap)        # black
            all_times.append(tval)

    # Convert to arrays
    all_indices = np.array(all_indices)
    all_norm_intensity = np.array(all_norm_intensity)
    all_times = np.array(all_times)

    print(np.unique(all_norm_intensity, return_counts=True))

    # -----------------------------------------
    # FINAL SINGLE-CALL VISUALIZATION
    # -----------------------------------------

    visualize_3D_data_event_for_detector(
        all_indices,
        all_norm_intensity,
        all_times,
        colorscale=colorscale,
        surface_color='lightgray',
    )

In [ ]:
event_idx = 0
visualize_event_all_particles(all_events[event_idx])